# ***1. Introduction***

## **Customer Analytics using PySpark RDD**

### Objective
The goal of this assignment is to understand and implement various RDD transformations in PySpark using a real-world dataset.

### Dataset Description
The dataset "customers_report" contains customer-level aggregated data including:
- Demographics (age, age_group)
- Transaction metrics (total_sales, total_orders)
- Behavioral metrics (recency, lifespan, avg_order_value)

### Learning Outcomes
- Understand RDD transformations (map, filter, reduceByKey, etc.)
- Learn lazy evaluation and DAG concepts
- Perform real-world customer analytics

# ***2. Setup & Data Loading***

In [1]:
from pyspark import SparkContext

sc = SparkContext()

# Load data
rdd = sc.textFile("customers_report.csv")

# Remove header
header = rdd.first()
data = rdd.filter(lambda x: x != header)

# Convert to structured format
rdd = data.map(lambda x: x.split(","))

In [2]:
# rdd.collect()

In [3]:
for row in rdd.take(10):
  print(row)

['1', 'AW00011000', 'Jon Yang', '54', '50 and above', 'VIP', '2013-05-03', '140', '3', '8249', '8', '8', '2749', '294']
['2', 'AW00011001', 'Eugene Huang', '49', '40-49', 'VIP', '2013-12-10', '133', '3', '6384', '11', '10', '2128', '182']
['3', 'AW00011002', 'Ruben Torres', '54', '50 and above', 'VIP', '2013-02-23', '143', '3', '8114', '4', '4', '2704', '324']
['4', 'AW00011003', 'Christy Zhu', '52', '50 and above', 'VIP', '2013-05-10', '140', '3', '8139', '9', '9', '2713', '280']
['5', 'AW00011004', 'Elizabeth Johnson', '46', '40-49', 'VIP', '2013-05-01', '140', '3', '8196', '6', '6', '2732', '292']
['6', 'AW00011005', 'Julio Ruiz', '49', '40-49', 'VIP', '2013-05-02', '140', '3', '8121', '6', '6', '2707', '280']
['7', 'AW00011006', 'Janet Alvarez', '49', '40-49', 'VIP', '2013-05-14', '140', '3', '8119', '5', '5', '2706', '289']
['8', 'AW00011007', 'Marco Mehta', '56', '50 and above', 'VIP', '2013-03-19', '142', '3', '8211', '8', '8', '2737', '315']
['9', 'AW00011008', 'Rob Verhoff', '

In [4]:
print(header)

customer_key,customer_number,customer_name,age,age_group,customer_segment,last_order_date,recency,total_orders,total_sales,total_quantity,lifespan,avg_order_value,avg_monthly_spend


# ***3. Beginner Level Transformations***

## *3.1 Filter Customers Age > 50*

### Transformation: filter()

We filter customers whose age is greater than 50.

In [5]:
age_filter = rdd.filter(lambda x: int(x[3]) > 50)

In [6]:
# age_filter.take(5)

In [7]:
for data in age_filter.take(10):
  print(data)

['1', 'AW00011000', 'Jon Yang', '54', '50 and above', 'VIP', '2013-05-03', '140', '3', '8249', '8', '8', '2749', '294']
['3', 'AW00011002', 'Ruben Torres', '54', '50 and above', 'VIP', '2013-02-23', '143', '3', '8114', '4', '4', '2704', '324']
['4', 'AW00011003', 'Christy Zhu', '52', '50 and above', 'VIP', '2013-05-10', '140', '3', '8139', '9', '9', '2713', '280']
['8', 'AW00011007', 'Marco Mehta', '56', '50 and above', 'VIP', '2013-03-19', '142', '3', '8211', '8', '8', '2737', '315']
['10', 'AW00011009', 'Shannon Carlson', '56', '50 and above', 'VIP', '2013-05-09', '140', '3', '8091', '5', '5', '2697', '288']
['11', 'AW00011010', 'Jacquelyn Suarez', '56', '50 and above', 'VIP', '2013-05-23', '140', '3', '8088', '4', '4', '2696', '288']
['12', 'AW00011011', 'Curtis Lu', '56', '50 and above', 'VIP', '2013-03-19', '142', '3', '8133', '4', '4', '2711', '301']
['15', 'AW00011014', 'Sydney Bennett', '52', '50 and above', 'New', '2013-04-30', '141', '2', '138', '6', '5', '69', '138']
['18', 

## *3.2 Extract Customer Name and Sales*

### Transformation: map()

Extracting specific columns (customer_name, total_sales).

In [8]:
name_sales = rdd.map(lambda x: (x[2], int(x[9])))

In [9]:
name_sales.take(5)

[('Jon Yang', 8249),
 ('Eugene Huang', 6384),
 ('Ruben Torres', 8114),
 ('Christy Zhu', 8139),
 ('Elizabeth Johnson', 8196)]

## *3.3 Unique Age Groups*

### Transformation: distinct()

Finding unique age groups in the dataset.

In [10]:
age_group = rdd.map(lambda x: x[4]).distinct()

In [11]:
age_group.collect()

['50 and above', '40-49', '30-39']

**Note:**
- Use .take(n) only when you need first n elements
- Use .collect() only when you need all the elements in RDD

## *3.4 flatMap()*

### Transformation: flatMap()

Split age_group into words

In [12]:
age_words_rdd = rdd.flatMap(lambda x: x[4].split(" "))

In [13]:
age_words_rdd.take(10)

['50', 'and', 'above', '40-49', '50', 'and', 'above', '50', 'and', 'above']

## *3.5 union()*

### union(): Combine two subsets of customers

In [14]:
young_rdd = rdd.filter(lambda x: x[3].isdigit() and int(x[3]) < 40)
senior_rdd = rdd.filter(lambda x: x[3].isdigit() and int(x[3]) >= 50)

combined_rdd = young_rdd.union(senior_rdd)

In [15]:
# combined_rdd.take(5)

In [16]:
for data in combined_rdd.take(5):
  print(data)

['50', 'AW00011049', 'Carol Rai', '39', '30-39', 'New', '2013-10-29', '135', '2', '81', '5', '5', '40', '11']
['54', 'AW00011053', 'Ana Price', '39', '30-39', 'New', '2013-01-17', '144', '1', '2330', '2', '2', '2330', '2330']
['133', 'AW00011132', 'Melissa Richardson', '39', '30-39', 'New', '2013-04-09', '141', '1', '2372', '6', '6', '2372', '2372']
['134', 'AW00011133', 'Angela Griffin', '39', '30-39', 'New', '2013-12-30', '133', '1', '72', '4', '4', '72', '72']
['474', 'AW00011473', 'Grace Henderson', '39', '30-39', 'New', '2013-06-02', '139', '1', '63', '2', '2', '63', '63']


# ***4. Intermediate Transformations***

## *4.1 reduceByKey()*

### reduceByKey(): Total sales per age group

In [17]:
sales_by_age_rdd = rdd.map(lambda x: (x[4], int(x[9]))).reduceByKey(lambda x, y: x + y)

In [18]:
sales_by_age_rdd.collect()

[('50 and above', 19505238), ('40-49', 9566103), ('30-39', 279917)]

## *4.2 groupByKey()*

### groupByKey(): Group customer names by segment

In [19]:
grouped_customers_rdd = rdd.map(lambda x: (x[5], x[2])).groupByKey()

In [20]:
grouped_customers_rdd.collect()

[('VIP', <pyspark.resultiterable.ResultIterable at 0x7b648ed8ac60>),
 ('Regular', <pyspark.resultiterable.ResultIterable at 0x7b648e416960>),
 ('New', <pyspark.resultiterable.ResultIterable at 0x7b648e417050>)]

## *4.3 sortBy()*

### sortBy(): Sort customers by total_sales

In [21]:
sorted_sales_rdd = rdd.sortBy(lambda x: int(x[9]), ascending = False)

In [22]:
# sorted_sales_rdd.take(5)

In [23]:
for data in sorted_sales_rdd.take(5):
  print(data)

['1133', 'AW00012132', 'Kaitlyn Henderson', '64', '50 and above', 'VIP', '2013-10-17', '135', '5', '13294', '14', '13', '2658', '402']
['1302', 'AW00012301', 'Nichole Nara', '73', '50 and above', 'VIP', '2013-11-20', '134', '5', '13294', '13', '11', '2658', '443']
['1309', 'AW00012308', 'Margaret He', '55', '50 and above', 'VIP', '2013-11-19', '134', '5', '13268', '14', '14', '2653', '457']
['1132', 'AW00012131', 'Randall Dominguez', '64', '50 and above', 'VIP', '2013-10-10', '135', '5', '13265', '11', '11', '2653', '414']
['1301', 'AW00012300', 'Adriana Gonzalez', '73', '50 and above', 'VIP', '2013-10-17', '135', '5', '13242', '10', '10', '2648', '456']


## *4.4 mapValue()*

### mapValues(): Increase avg_order_value by 10%

In [24]:
inc_value_rdd = rdd.map(lambda x: (x[2], int(x[12]))).mapValues(lambda x: x * 1.1)

In [25]:
# inc_value_rdd.collect()

In [26]:
for data in inc_value_rdd.take(10):
  print(data)

('Jon Yang', 3023.9)
('Eugene Huang', 2340.8)
('Ruben Torres', 2974.4)
('Christy Zhu', 2984.3)
('Elizabeth Johnson', 3005.2000000000003)
('Julio Ruiz', 2977.7000000000003)
('Janet Alvarez', 2976.6000000000004)
('Marco Mehta', 3010.7000000000003)
('Rob Verhoff', 2972.2000000000003)
('Shannon Carlson', 2966.7000000000003)


## *4.5 keys() and values()*

### keys() and values(): Extract keys and values separately

In [27]:
kv_rdd = rdd.map(lambda x: (x[2], int(x[9])))

In [28]:
keys_rdd = kv_rdd.keys()
values_rdd = kv_rdd.values()

In [29]:
keys_rdd.take(5)

['Jon Yang',
 'Eugene Huang',
 'Ruben Torres',
 'Christy Zhu',
 'Elizabeth Johnson']

In [30]:
for data in keys_rdd.take(5):
  print(data)

Jon Yang
Eugene Huang
Ruben Torres
Christy Zhu
Elizabeth Johnson


In [31]:
values_rdd.take(5)

[8249, 6384, 8114, 8139, 8196]

In [32]:
for data in values_rdd.take(5):
  print(data)

8249
6384
8114
8139
8196


## *4.6 sortByKey()*

### sortByKey(): Sort segment-wise sales

In [33]:
sorted_by_segment_rdd = rdd.map(lambda x: (x[5], int(x[9]))).sortByKey()

In [34]:
sorted_by_segment_rdd.take(5)

[('New', 81), ('New', 114), ('New', 138), ('New', 2501), ('New', 2332)]

In [35]:
for data in sorted_by_segment_rdd.take(5):
  print(data)

('New', 81)
('New', 114)
('New', 138)
('New', 2501)
('New', 2332)


## *4.7 intersection()*

### intersection(): Common customers in two conditions

In [36]:
# high_sales_rdd = rdd.filter(lambda x: int(x[9]) > 8000)
# vip_rdd = rdd.filter(lambda x: x[5] == "VIP")

# common_rdd = high_sales_rdd.intersection(vip_rdd)
# Showing Error

In [37]:
high_sales_rdd = rdd.filter(lambda x: x[9].isdigit() and int(x[9]) > 8000).map(tuple)
VIP_rdd = rdd.filter(lambda x: x[5] == "VIP").map(tuple)

In [38]:
common_rdd = high_sales_rdd.intersection(VIP_rdd)

In [39]:
# common_rdd.take(5)

In [40]:
for data in common_rdd.take(5):
  print(data)

('55', 'AW00011054', 'Deanna Munoz', '68', '50 and above', 'VIP', '2013-05-30', '140', '3', '8223', '6', '6', '2741', '304')
('56', 'AW00011055', 'Gilbert Raje', '68', '50 and above', 'VIP', '2013-06-07', '139', '3', '8168', '8', '8', '2722', '281')
('73', 'AW00011072', 'Casey Luo', '65', '50 and above', 'VIP', '2013-06-15', '139', '3', '8194', '7', '7', '2731', '303')
('77', 'AW00011076', 'Blake Anderson', '62', '50 and above', 'VIP', '2013-06-21', '139', '3', '8159', '4', '4', '2719', '313')
('100', 'AW00011099', 'Adam Ross', '59', '50 and above', 'VIP', '2013-07-01', '138', '3', '8216', '9', '8', '2738', '304')


## *4.8 subtract()*

### subtract(): Remove sampled subset from dataset

In [41]:
subset_rdd = rdd.sample(False, 0.1)
# Taking a random 10% slice of data to test logic or perform quick analysis.

In [42]:
for data in subset_rdd.take(5):
  print(data)

['3', 'AW00011002', 'Ruben Torres', '54', '50 and above', 'VIP', '2013-02-23', '143', '3', '8114', '4', '4', '2704', '324']
['11', 'AW00011010', 'Jacquelyn Suarez', '56', '50 and above', 'VIP', '2013-05-23', '140', '3', '8088', '4', '4', '2696', '288']
['14', 'AW00011013', 'Ian Jenkins', '46', '40-49', 'New', '2014-01-21', '132', '2', '114', '5', '5', '57', '12']
['17', 'AW00011016', 'Wyatt Hill', '41', '40-49', 'New', '2013-02-09', '143', '1', '2332', '3', '3', '2332', '2332']
['20', 'AW00011019', 'Luke Lal', '42', '40-49', 'New', '2014-01-12', '132', '17', '880', '33', '20', '51', '80']


In [43]:
remaining_rdd = rdd.map(tuple).subtract(subset_rdd.map(tuple))

In [44]:
# remaining_rdd.collect()

In [45]:
for data in remaining_rdd.take(15):
  print(data)

('2', 'AW00011001', 'Eugene Huang', '49', '40-49', 'VIP', '2013-12-10', '133', '3', '6384', '11', '10', '2128', '182')
('28', 'AW00011027', 'Jessie Zhao', '73', '50 and above', 'VIP', '2013-10-24', '135', '3', '6591', '9', '9', '2197', '199')
('29', 'AW00011028', 'Jill Jimenez', '74', '50 and above', 'VIP', '2013-10-07', '135', '3', '6474', '5', '5', '2158', '196')
('37', 'AW00011036', 'Jennifer Russell', '41', '40-49', 'New', '2013-01-22', '144', '1', '2355', '2', '2', '2355', '2355')
('40', 'AW00011039', 'Marc Martin', '71', '50 and above', 'VIP', '2013-11-11', '134', '3', '6636', '9', '8', '2212', '201')
('46', 'AW00011045', 'Leonard Nara', '70', '50 and above', 'New', '2013-10-29', '135', '2', '115', '4', '4', '57', '19')
('47', 'AW00011046', 'Christine Yuan', '70', '50 and above', 'VIP', '2013-11-12', '134', '3', '6616', '6', '6', '2205', '200')
('48', 'AW00011047', 'Jaclyn Lu', '70', '50 and above', 'VIP', '2013-11-19', '134', '3', '6582', '4', '4', '2194', '199')
('49', 'AW00011

# ***5. Advanced Transformations***

## *5.1 combineByKey()*

### combineByKey(): Average sales per segment

In [46]:
avg_sales_rdd = rdd.map(lambda x: (x[4], int(x[9]))).combineByKey(
    lambda x: (x, 1),
    lambda acc, x: (acc[0] + x, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
).mapValues(lambda x: round(x[0] / x[1], 2))

In [47]:
avg_sales_rdd.collect()

[('50 and above', 1601.02), ('40-49', 1567.44), ('30-39', 1428.15)]

In [48]:
for data in avg_sales_rdd.collect():
  print(data)

('50 and above', 1601.02)
('40-49', 1567.44)
('30-39', 1428.15)


## *5.2 mapPartitions()*

### mapPartitions(): Process data at partition level

In [49]:
def partition_func(iterator):
  return [(x[2], int(x[9])) for x in iterator]

In [50]:
partition_rdd = rdd.mapPartitions(partition_func)

In [51]:
# partition_rdd.collect()

In [52]:
for data in partition_rdd.take(10):
  print(data)

('Jon Yang', 8249)
('Eugene Huang', 6384)
('Ruben Torres', 8114)
('Christy Zhu', 8139)
('Elizabeth Johnson', 8196)
('Julio Ruiz', 8121)
('Janet Alvarez', 8119)
('Marco Mehta', 8211)
('Rob Verhoff', 8106)
('Shannon Carlson', 8091)


## *5.3 repartitions()*

### repartition(): Increase number of partitions

In [53]:
repartitioned_rdd = rdd.repartition(4)

In [54]:
repartitioned_rdd.getNumPartitions()

4

## *5.4 coalesce()*

### coalesce(): Reduce number of partitions

In [55]:
coalesced_rdd = rdd.coalesce(2)

In [56]:
coalesced_rdd.getNumPartitions()

2